In [ ]:
import numpy as np
import pandas as pd
import joblib
from app.features.tabular import engineer_tab_features

def predict_df(df: pd.DataFrame, *, artifacts=None, model_path: str|None=None) -> pd.DataFrame:
    """
    Adiciona colunas proba_H (float) e pred (0/1) ao df, usando o mesmo pré-processamento do treino.
    Passe artifacts (dict carregado em memória) OU model_path (joblib).
    """
    if artifacts is None:
        if not model_path:
            raise ValueError("Forneça artifacts ou model_path.")
        artifacts = joblib.load(model_path)

    emb_pipe = artifacts["embedding_pipeline"]
    tab_pipe = artifacts["tabular_pipeline"]
    model = artifacts["model"]
    best_thr = artifacts.get("best_threshold", 0.5)
    tab_cfg = artifacts.get("tabular_config", {})
    tab_mode = tab_cfg.get("tab_mode", "latlon_time")

    # Embeddings -> matriz 2D
    embedding_matrix = np.vstack(df["embedding"].values)
    emb_cols = [f"emb_{i}" for i in range(embedding_matrix.shape[1])]
    df_embs = pd.DataFrame(embedding_matrix, columns=emb_cols, index=df.index)

    # Features tabulares (usa a mesma config salva no treino, incl. IBGE)
    df_tab, _ = engineer_tab_features(
        df,
        mode=tab_mode,
        urban_areas_path=tab_cfg.get("urban_areas_path"),
        urban_layer=tab_cfg.get("urban_layer"),
        urban_radius_km=tab_cfg.get("urban_radius_km", 5.0),
    )

    # Transformar e prever
    X_emb = emb_pipe.transform(df_embs)
    X_tab = tab_pipe.transform(df_tab)
    X = np.hstack([X_emb, X_tab])
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= best_thr).astype(int)

    out = df.copy()
    out["proba_H"] = proba
    out["pred"] = pred
    return out

# Exemplo de uso (se você quiser evitar carregar arquivos, passe artifacts já em memória)
# artifacts = joblib.load("outputs/final_model_pca/final_model.joblib")  # ou já fornecido
# df_preds = predict_df(df, artifacts=artifacts)

In [3]:
artifacts = joblib.load("outputs/final_model_pca/final_model.joblib")

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/final_model_pca/final_model.joblib'